[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/tarekFerdous/python-practice/blob/main/exercises/0002-two-pointers-sliding-window/exercises.ipynb)

# Lesson 2 — Two pointers and the sliding window

Lesson: [0002-two-pointers-sliding-window.html](https://github.com/tarekFerdous/python-practice/blob/main/lessons/0002-two-pointers-sliding-window.html) (open the HTML locally for the quizzes)

**How to use:** run the setup cell once. Then for each exercise: edit the function cell, run it (`Shift+Enter`), then run the check cell under it. Each check prints PASS/FAIL with what you returned vs what was expected. `Run All` checks everything.

Before coding each one, say out loud: brute force + its Big-O, then your idea. When it passes, give **time and space** complexity.

In [ ]:
# Setup: run this cell first (test helper, same as assets/nbcheck.py)
"""Tiny test runner for the exercise notebooks.

    check(two_sum, [
        (([2, 7, 11, 15], 9), (0, 1)),   # (args tuple, expected)
    ])

Prints one line per case: what you called, what you got, what was expected.
Exceptions are caught and shown instead of stopping the notebook.
"""
import copy

GREEN, RED, RESET = "\033[32m", "\033[31m", "\033[0m"


def check(fn, cases, name=None):
    name = name or getattr(fn, "__qualname__", repr(fn))
    passed = 0
    for args, expected in cases:
        if not isinstance(args, tuple):
            args = (args,)
        shown = ", ".join(repr(a) for a in args)
        try:
            got = fn(*copy.deepcopy(args))
        except Exception as e:  # show the crash, keep going
            got = f"<{type(e).__name__}: {e}>"
        ok = got == expected and type(got) is type(expected)
        passed += ok
        mark = f"{GREEN}PASS{RESET}" if ok else f"{RED}FAIL{RESET}"
        line = f"{mark}  {name}({shown}) -> {got!r}"
        if not ok:
            line += f"   expected {expected!r}"
        print(line)
    colour = GREEN if passed == len(cases) else RED
    print(f"{colour}{passed}/{len(cases)} passed{RESET}\n")
    return passed == len(cases)

## 1. `is_palindrome`

In [ ]:
def is_palindrome(s: str) -> bool:
    """True if s reads the same both ways, ignoring case and anything that isn't a letter or digit.
    Use two pointers from both ends. O(1) extra space: don't build a cleaned copy.

    is_palindrome("A man, a plan, a canal: Panama") -> True
    is_palindrome("race a car")                     -> False
    """
    s = s.lower()
    s = "".join(c for c in s if c.isalnum())
    # print(s)
    left, right = 0, len(s)-1
    counter = 0
    while left<right:
        if s[left] != s[right]:
            return False
        else:
            left +=1
            right -=1
    return True

In [ ]:
check(is_palindrome, [
    ("A man, a plan, a canal: Panama", True),
    ("race a car", False),
    ("", True),
    (" .,", True),
    ("0P", False),
    ("ab_a", True),
])

## 2. `pair_with_sum`

In [ ]:
def pair_with_sum(nums: list[int], target: int) -> tuple[int, int] | None:
    """nums is sorted ascending. Return the indices (i, j), i < j, of two numbers that add up to
    target, or None if there are none. At most one pair adds up to target.
    O(n) time and O(1) space: no dict this time.

    pair_with_sum([1, 3, 4, 6, 9], 13) -> (2, 4)    # 4 + 9
    pair_with_sum([1, 2, 3], 100)      -> None
    """
    left, right = 0, len(nums)-1
    while left < right:
        if nums[left] + nums[right] == target:
            return (left, right)
        elif nums[left] + nums[right] < target:
            left +=1
        else:
            right +=1
        
    return None

In [ ]:
check(pair_with_sum, [
    (([1, 3, 4, 6, 9], 13), (2, 4)),
    (([1, 2, 3], 100), None),
    (([-5, -1, 0, 7], 2), (0, 3)),
    (([2, 2], 4), (0, 1)),
    (([], 1), None),
    (([1, 2, 3, 10], 5), (1, 2)),   # needs the 'too big' branch
])

## 3. `max_sum_of_k`

In [ ]:
def max_sum_of_k(nums: list[int], k: int) -> int:
    """Largest sum of any k consecutive numbers. Assume 1 <= k <= len(nums).
    Fixed-size window: O(n), not O(n * k).

    max_sum_of_k([2, 1, 5, 1, 3, 2], 3) -> 9        # 5 + 1 + 3
    """
    k_reached = 0
    sum = 0
    maxSum = 0
    
    for i, v in enumerate(nums):
        if k_reached<k:
            sum = sum + v
            maxSum = sum
        else:
            sum = sum + v - nums[i-k]
        

        k_reached += 1
        
        if sum > maxSum and k_reached >= k:
            maxSum = sum
    return maxSum

In [ ]:
check(max_sum_of_k, [
    (([2, 1, 5, 1, 3, 2], 3), 9),
    (([4], 1), 4),
    (([-3, -1, -2], 2), -3),
    (([1, 2, 3, 4], 4), 10),
])

## 4. `longest_unique_substring`

In [ ]:
def longest_unique_substring(s: str) -> int:
    """Length of the longest substring with no repeated characters.
    Variable-size window: grow on the right, shrink from the left when a repeat appears.

    longest_unique_substring("abcabcbb") -> 3       # "abc"
    longest_unique_substring("pwwkew")   -> 3       # "wke"
    """
    last_seen = {}  # char -> index where it was last seen
    left = 0        # start of the current window (no repeats inside it)
    best = 0
    for right, c in enumerate(s):
        # If c already appears inside the window, move left just past that copy.
        if c in last_seen and last_seen[c] >= left:
            left = last_seen[c] + 1
        last_seen[c] = right
        best = max(best, right - left + 1)
    return best

In [ ]:
check(longest_unique_substring, [
    ("abcabcbb", 3),
    ("bbbbb", 1),
    ("pwwkew", 3),
    ("", 0),
    ("abba", 2),
    ("dvdf", 3),
])

---
## Teacher's idiomatic versions
Read after yours pass, compare, then rewrite from memory.

In [ ]:
from collections import Counter, defaultdict
import heapq

In [ ]:
def is_palindrome_v2(s: str) -> bool:
    # O(n) time, O(1) space: skip junk by moving one pointer, no cleaned copy
    left, right = 0, len(s) - 1
    while left < right:
        if not s[left].isalnum():
            left += 1
        elif not s[right].isalnum():
            right -= 1
        elif s[left].lower() != s[right].lower():
            return False
        else:
            left += 1
            right -= 1
    return True

In [ ]:
check(is_palindrome_v2, [
    ("A man, a plan, a canal: Panama", True),
    ("race a car", False),
    ("", True),
    (" .,", True),
    ("0P", False),
    ("ab_a", True),
])

In [ ]:
def max_sum_of_k_v2(nums: list[int], k: int) -> int:
    # O(n) time, O(1) space
    window = sum(nums[:k])
    best = window
    for right in range(k, len(nums)):
        window += nums[right] - nums[right - k]   # in one, out one
        best = max(best, window)
    return best

In [ ]:
check(max_sum_of_k_v2, [
    (([2, 1, 5, 1, 3, 2], 3), 9),
    (([4], 1), 4),
    (([-3, -1, -2], 2), -3),
    (([1, 2, 3, 4], 4), 10),
])